# MR-BT

This file gives a template of how to **compute (per-validation-datum) FreeShap values** of each datum for finetuning a BERT model on the Movie Review dataset and **evaluate** a selected MR subset. Note that the evaluation would take some time.

In [1]:
from main.utils.gpu import use_gpus_, connect_to_

use_gpus_([0])
device = connect_to_(0)

Connected to cuda:0


In [2]:
configs = {
    "yaml_path": "main/configs/mr-bert_ntk.yaml",
    "dataset_name": "mr",
    "file_path": "saved_data/mr-bert/",
    "num_dp": 8530,
    "tmc_iter": 1000,
    "approximate": 'inv',
    "early_stopping": True,
    "parallel": 40,
    "seed": 2023
}

In [3]:
from main.shapley.freeshap import free_shapley

shapleys, ps_shapleys = free_shapley(**configs)
ps_shapleys.shape

(8530, 1066)

In [4]:
import numpy as np
from main.evaluate.mr import evaluate_mr_subset

n_selected = 1000
indices = np.argsort(shapleys)[::-1][:n_selected]

results = evaluate_mr_subset(indices, device=device, seed=configs["seed"])
print(results)

Constructing <class 'dataset.EasyReader'>
Constructing <class 'dataset.ListDataset'>


/home/tianxiao/miniconda3/envs/S25001/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Label 0 to word terrible (6659)
Label 1 to word great (2307)
label_to_word:  {0: 6659, 1: 2307}
label_list:  [6659, 2307]
Constructing <class 'probe.PromptFinetuneProbe'>
Constructing PromptFinetuneProbe


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Probe has 109514298 parameters


...

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Probe has 109514298 parameters
Probe has 109515836 parameters


/home/tianxiao/miniconda3/envs/S25001/lib/python3.10/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
50,0.264400
100,0.124900
150,0.094900
200,0.047000
250,0.031100
300,0.014700
350,0.010100
400,0.000900
450,0.007000
500,0.005100


{'eval_loss': 1.0286288261413574, 'eval_accuracy': 83.67729831144464, 'eval_runtime': 0.7846, 'eval_samples_per_second': 1358.626, 'eval_steps_per_second': 21.667, 'epoch': 10.0}
